# 03 — Data Preprocessing
Phase 4: a leakage-safe preprocessing pipeline. Train/test split happens **before** any fitting of imputers, scalers, or encoders.

In [1]:
import sys
sys.path.append('../src')
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from preprocessing import (load_raw_data, basic_clean, encode_target,
                            get_column_groups, build_preprocessing_pipeline, get_feature_names)

RANDOM_STATE = 42
df = load_raw_data('../data/raw/insurance_claims.csv')
df = basic_clean(df)
df = encode_target(df)
df.shape

(1000, 39)

## Why train/test split comes first
If we imputed missing values, scaled numeric features, or one-hot encoded categories using statistics from the **entire** dataset, information from the test set (e.g. the mean used for scaling) would leak into training. That inflates evaluation metrics and gives an optimistic, unrealistic picture of how the model will perform on genuinely unseen claims. So the split happens immediately, and every subsequent `fit` is on the training split only.

In [2]:
train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['fraud_reported'], random_state=RANDOM_STATE
)
print("Train shape:", train_df.shape, " Test shape:", test_df.shape)
print("Train fraud rate:", train_df['fraud_reported'].mean().round(3))
print("Test fraud rate:", test_df['fraud_reported'].mean().round(3))

Train shape: (800, 39)  Test shape: (200, 39)
Train fraud rate: 0.248
Test fraud rate: 0.245


Stratified split keeps the fraud rate consistent between train and test — important given the moderate class imbalance.

## Outlier handling — `umbrella_limit`
Notebook 01 flagged an implausible negative value (-1,000,000) in `umbrella_limit`. Rather than deleting rows (losing data), we clip to a plausible non-negative floor. This is applied **after** the split, computed only where relevant, and is not a data-leakage risk since 0 is a fixed, data-independent floor, not a statistic learned from the data.

In [3]:
for name, d in [('train', train_df), ('test', test_df)]:
    n_bad = (d['umbrella_limit'] < 0).sum()
    print(f"{name}: {n_bad} rows with negative umbrella_limit")

train_df['umbrella_limit'] = train_df['umbrella_limit'].clip(lower=0)
test_df['umbrella_limit'] = test_df['umbrella_limit'].clip(lower=0)

train: 1 rows with negative umbrella_limit
test: 0 rows with negative umbrella_limit


## Missing values, categorical encoding, scaling
Handled together via a scikit-learn `ColumnTransformer` (see `src/preprocessing.py`):
- **Numeric columns:** median imputation (robust to skew/outliers) + `StandardScaler` (needed for Logistic Regression; harmless for tree models).
- **Categorical columns:** most-frequent imputation for the small number of genuine missing values + `OneHotEncoder(handle_unknown='ignore')` so an unseen category at prediction time doesn't crash the pipeline.

We fit this transformer on the training split only, illustrated below on the raw (pre-feature-engineered) columns; the actual modeling pipeline in notebook 05 fits it on the feature-engineered training data.

In [4]:
numeric_cols, categorical_cols = get_column_groups(train_df.drop(columns=[
    'policy_number','insured_zip','incident_location','policy_bind_date','incident_date'
]))
print(f"{len(numeric_cols)} numeric columns, {len(categorical_cols)} categorical columns")

preprocessor = build_preprocessing_pipeline(numeric_cols, categorical_cols)
X_train_demo = preprocessor.fit_transform(train_df[numeric_cols + categorical_cols])
X_test_demo = preprocessor.transform(test_df[numeric_cols + categorical_cols])
print("Encoded shapes:", X_train_demo.shape, X_test_demo.shape)

16 numeric columns, 17 categorical columns
Encoded shapes: (800, 157) (200, 157)


## Class imbalance and data leakage — previewed here, applied in notebook 05
- **Class imbalance** (24.7% fraud) is handled with SMOTE, applied to the **training split only**, after the train/test split and after preprocessing — never to the test set, and never before the split. Oversampling the whole dataset before splitting would let synthetic points derived from test-set claims leak into training.
- **Data leakage checklist used throughout this project:**
  - Split before any fitting ✅
  - Imputer/scaler/encoder fit on train only ✅
  - SMOTE applied to train only ✅ (notebook 05)
  - Feature engineering thresholds (e.g. the 75th-percentile "high value claim" cutoff) computed on train only, then reused on test ✅ (notebook 04)
  - No feature derived from the fraud outcome or post-investigation data is used ✅

## Phase 4 summary
The dataset needed: stratified splitting first, a small structural-missingness imputation strategy, one clear outlier fix (`umbrella_limit`), one-hot encoding for 16 categorical columns, and scaling for numeric columns — all wired into a single reusable `ColumnTransformer` object saved for production scoring in `src/predict.py`.